# Note

This code is based on the one developed by Gilad Wisney in:

https://towardsdatascience.com/deep-reinforcement-learning-and-monte-carlo-tree-search-with-connect-4-ba22a4713e7a

https://github.com/giladariel/Connect4

# Imports

In [1]:
import numpy as np

import random
import pickle

import keras.layers as Kl
import keras.models as Km
import tensorflow
from tensorflow import keras
from keras import optimizers

import copy

import os
from pathlib import Path

import datetime

# Seed

In [2]:
#The seeds are randomly generated numbers from 1000 to 99999
seeds1 = [66914, 79634, 5956, 64082, 19873, 25073, 51152, 56884, 91580, 42875]
seeds2 = [68662, 65498, 94139, 72143, 50989, 27377, 71519, 67051, 90895, 14652]
seeds3 = [37545, 81792, 55010, 36090, 89100, 47606, 43480, 1129, 54760, 75158]
#seeds = seeds1 + seeds2 + seeds3
#seeds = seeds2
seeds = [seeds2[0]]
seedsSize = len(seeds)

In [3]:
random.seed( seeds1[0] )

# Connect Four

In [4]:
class ConnectFour:

    def __init__(self, player1, player2, episodes, learn, printGame):
        self.players = {1: player1,
                        2: player2}

        self.state, self.winner, self.turn = self.init_game()
        self.memory = {}
        
        self.episodes = episodes+1
        self.learn = learn
        self.printGame = printGame
        
    def init_game(self):
        return np.zeros((6, 7)), None, 1    
        
    def play_multiple_games(self):
        statistics = {1: 0, 2: 0, 0: 0, "Number of Moves": 0}
        move_count_total = []
        for episode in range(1, self.episodes):
            if (self.printGame):
                print( "Game " + str(episode) )
            winner, move_count = self.play_game(episode)
            move_count_total.append(move_count)
            statistics[winner] = statistics[winner] + 1

            self.state, self.winner, self.turn = self.init_game()

        #Save the models
        if self.learn is True and isinstance(self.players[1], MCTSAgent):
            self.players[1].save_tree()
        if self.learn is True and isinstance(self.players[2], MCTSAgent):
            self.players[2].save_tree()
        
        statistics["Number of Moves"] = np.mean(move_count_total)
        return statistics

    def play_game(self, episode):
        move_count = 0

        while self.winner is None:
            player = self.players[self.turn]
            move = player.choose_move(self.state, self.winner, self.learn, episode)

            self.state = self.make_state_from_move(move)
            self.game_winner()

            self.next_player()
            move_count += 1

        self.play_move(episode)
        self.next_player()
        self.play_move(episode)
        self.next_player()
            
        return self.winner, move_count
    
    def play_move(self, episode):
        player = self.players[self.turn]
        move = player.choose_move(self.state, self.winner, self.learn, episode)
        return move
    
    def make_state_from_move(self, move):
        if move is None: #Remove?
            return self.state

        state = np.array(self.state)

        if self.turn == 1:
            tag = 1
        else:
            tag = -1

        row = np.where(state[:, move] == 0)[0][-1] #Get the first empty space of the column (from the bottom)

        state[row, move] = tag #Fill that space

        return state

    def next_player(self):
        if self.turn == 1:
            self.turn = 2
        else:
            self.turn = 1

    def game_winner(self):
        for i in range(len(self.state[:,0])-3): #Rows
            for j in range(len(self.state[0, :])-3): #Columns
                self.square_winner(self.state[i:i+4, j:j+4]) #Check Winner for the 4x4 grid
                if self.winner is not None:
                    break
            if self.winner is not None:
                if (self.printGame): 
                    print('winner is:', self.winner)
                    print(self.state)
                break

        if np.min(np.abs(self.state[0, :])) != 0: #Check if first row is full
            self.winner = 0
            if (self.printGame): 
                print('no winner')
                print(self.state)

    def square_winner(self, square):
        #np.sum(square, axis=0) -> Sum along each column
        #np.sum(square, axis=1) -> Sum along each row
        #np.trace(square) -> Sum along the diagonal
        #np.flip(square,axis=1).trace() -> Sum along the other diagonal
        s = np.append([np.sum(square, axis=0), np.sum(square, axis=1)],
                      [np.trace(square), np.flip(square,axis=1).trace()])
        if np.max(s) == 4:
            self.winner = 1
        elif np.min(s) == -4:
            self.winner = 2
        else:
            self.winner = None
        return self.winner

# Available Moves

In [5]:
def ava_moves(state):
    moves = np.where(state[0, :] == 0)[0]
    return moves

# First or Second

In [6]:
def firstOrSecond(tag):
    if tag == 1:
        return 'First'
    else:
        return 'Second'

# Connect 4 Model

In [7]:
class ConnectFourModel:

    def __init__(self, tag, network):
        self.tag = tag
        self.epsilon = 0.1
        self.alpha = 0.5
        self.gamma = 1
        self.network = network
        self.model = self.load_model()
        
    def create_model(self):
        print('New Model - Network ' + self.network)

        if self.network == "A":
            model = self.model_A()
        elif self.network == "B":
            model = self.model_B()
            
        model.summary()
        
        return model
    
    def model_A(self):
        model = Km.Sequential()
        model.add(Kl.Conv2D(20, (5, 5), padding='same', input_shape=(7, 7, 1)))
        model.add(Kl.LeakyReLU(alpha=0.3))
        model.add(Kl.Conv2D(20, (4, 4), padding='same'))
        model.add(Kl.LeakyReLU(alpha=0.3))
        model.add(Kl.Conv2D(20, (4, 4), padding='same'))
        model.add(Kl.LeakyReLU(alpha=0.3))
        model.add(Kl.Conv2D(30, (4, 4), padding='same'))
        model.add(Kl.LeakyReLU(alpha=0.3))
        model.add(Kl.Conv2D(30, (4, 4), padding='same'))
        model.add(Kl.LeakyReLU(alpha=0.3))
        model.add(Kl.Conv2D(30, (4, 4), padding='same'))
        model.add(Kl.LeakyReLU(alpha=0.3))
        model.add(Kl.Conv2D(30, (4, 4), padding='same'))
        model.add(Kl.LeakyReLU(alpha=0.3))

        model.add(Kl.Flatten(input_shape=(7, 7, 1)))
        model.add(Kl.Dense(49))
        model.add(Kl.LeakyReLU(alpha=0.3))
        model.add(Kl.Dense(7))
        model.add(Kl.LeakyReLU(alpha=0.3))
        model.add(Kl.Dense(7))
        model.add(Kl.LeakyReLU(alpha=0.3))

        model.add(Kl.Dense(1, activation='linear'))
        opt = tensorflow.optimizers.Adam(learning_rate=0.001)
        model.compile(optimizer=opt, loss='mean_squared_error', metrics=['accuracy'])

        return model
    
    def model_B(self): #Adapted from https://github.com/codebox/connect4/blob/master/src/nn/network_128x4_64_64.py
        model = Km.Sequential()
        model.add(Kl.Conv2D(128, (4,4), input_shape=(7, 7, 1)))
        model.add(Kl.Activation('relu'))

        model.add(Kl.Flatten())
        model.add(Kl.Dense(64))
        model.add(Kl.Activation('relu'))
        model.add(Kl.Dense(64))
        model.add(Kl.Activation('relu'))
        model.add(Kl.Dense(1))

        opt = keras.optimizers.Adam()

        model.compile(loss='mean_squared_error',
                      optimizer=opt,
                      metrics=['accuracy'])
        
        return model
    
    def load_model(self):
        tag = firstOrSecond(self.tag)
        
        s = 'Q-Learning Agents/Q_Learning_Agent_Going_' + tag + '.h5'
        model_file = Path(s)

        if model_file.is_file():
            model = Km.load_model(s)
            print('load model: ' + s)
        else:
            model = self.create_model()
        return model
        
    def save_model(self):
        tag = firstOrSecond(self.tag)
        
        s = 'Q-Learning Agents/Q_Learning_Agent_Going_' + tag + '.h5'

        try:
            os.remove(s)
        except:
            pass

        self.model.save(s)    

    def state_to_tensor(self, state, move):
        vec = np.zeros((1, 7))
        if move != -1:
            vec[0, move] = 1 #Vector with 1 in the position of the column chosen
        tensor = np.append(vec, state, axis=0) #Append the state to the vector
        tensor = tensor.reshape((1, 7, 7, 1))

        return tensor
    
    def calc_value(self, state, move):
        tensor = self.state_to_tensor(state, move)
        value = self.model.predict(tensor)
        return value

    def calc_target(self, prev_state, prev_move, state, available_moves, reward):
        v_s = self.calc_value(prev_state, prev_move)

        v = []
        for move in available_moves:
            v.append(self.calc_value(state, move))

        if reward == 0:
            if len(v) > 0:
                v_s_tag = self.gamma * np.max(v)
            else:
                print('no moves!!!')
                v_s_tag = 0
            target = np.array(v_s + self.alpha * (reward + v_s_tag - v_s))
        else:
            target = reward

        return target

    def train_model(self, prev_state, prev_move, target, epochs):
        tensor = self.state_to_tensor(prev_state, prev_move)

        if target is not None:
            # if self.tag == 1:
            #     print('value before training:', self.model.predict(tensor))
            self.model.fit(tensor, target, epochs=epochs, verbose=0)
            # if self.tag == 1:
            #     print('target:', target)
            #     print('value after training:', self.model.predict(tensor))
            
    def learn_batch(self, memory):
        print('start learning player', self.tag)
        print('data length:', len(memory))

        # build x_train
        ind = 0
        x_train = np.zeros((len(memory), 7, 7, 1))
        for v in memory:
            [prev_state, prev_move, _, _, _] = v
            sample = self.state_to_tensor(prev_state, prev_move)
            x_train[ind, :, :, :] = sample
            ind += 1

        # train with planning
        loss = 20
        count = 0
        while loss > 0.02:
            y_train = self.create_targets(memory)
            self.model.fit(x_train, y_train, epochs=5, batch_size=256, verbose=0)
            loss = self.model.evaluate(x_train, y_train, batch_size=256, verbose=0)[0]
            count += 1
            print('planning number:', count, 'loss', loss)

        loss = self.model.evaluate(x_train, y_train, batch_size=256, verbose=0)

        self.save_model()

    def create_targets(self, memory):
        y_train_ = np.zeros((len(memory), 1))
        count_ = 0
        for v_ in memory:
            [prev_state_, prev_move_, state_, available_moves_, reward_] = v_
            target = self.calc_target(prev_state_, prev_move_, state_, available_moves_, reward_)
            y_train_[count_, :] = target
            count_ += 1

        return y_train_

# Connect 4 Agent

In [8]:
class ConnectFourAgent:

    def __init__(self, tag, starting_epsilon, constant_epsilon, max_memory, network):
        self.tag = tag
        self.epsilon = starting_epsilon
        self.constant_epsilon=constant_epsilon
        self.prev_move = -1
        self.state = None
        self.move = None
        self.memory = []
        self.count_memory = 0
        self.max_memory = max_memory
        
        self.model = ConnectFourModel(tag, network)
        self.prev_state = np.zeros((6, 7))
        
    def choose_move(self, state, winner, learn, episode):
        self.load_to_memory(self.prev_state, self.prev_move, state, ava_moves(state), self.reward(winner))
        
        if winner is not None: #End of an episode
            self.count_memory += 1

            self.prev_state = np.zeros((6, 7))
            self.prev_move = -1
            
            #if (not self.constant_epsilon): #epsilon is not constant
                #self.epsilon = self.epsilon/((episode/100) + 1) #Update epsilon
                
            #print( "epsilon = " + str(self.epsilon) ) #Check if epsilon is updating

            if learn is True and self.count_memory == self.max_memory:
                self.count_memory = 0
                # Offline training
                self.model.learn_batch(self.memory)
                self.memory = []
                # Online training
                # self.learn(self.prev_state, self.prev_move, state, ava_moves(state),  -1, self.reward(winner))
            return None

        #if (self.constant_epsilon):
        column = self.epsilon_greedy(state)
        #else:
            #column = self.epsilon_greedy_2(state)

        self.prev_state = state
        self.prev_move = column

        return column
    
    def epsilon_greedy(self, state):
        p = random.uniform(0, 1)
        
        if p < self.epsilon: #Random
            #print("random move")
            available_moves = ava_moves(state)
            return random.choice(available_moves)
        else: #Optimal
            #print("optimal move")
            return self.choose_optimal_move(state) 
    
    def epsilon_greedy_2(self, state):
        p = random.uniform(0, 1)
        
        #print( "value = " + str((self.epsilon/7)+(1-self.epsilon)) ) #Check if epsilon is updating
        
        if p < ((self.epsilon/7)+(1-self.epsilon)): #((epsilon/n_actions)+(1-epsilon)) -> Optimal
            #print("optimal move")
            return self.choose_optimal_move(state) 
        else: #Random
            #print("random move")
            available_moves = ava_moves(state)
            return random.choice(available_moves)

    def choose_optimal_move(self, state):
        available_moves = ava_moves(state)
        v = -float('Inf')
        v_list = []
        column = []
        for move in available_moves:
            value = self.model.calc_value(state, move)
            v_list.append(round(float(value), 5))

            if value > v:
                v = value
                column = [move]
            elif v == value:
                column.append(move)

        column = random.choice(column)
        return column

    def reward(self, winner):
        if winner is self.tag:
            reward = 1
        elif winner is None:
            reward = 0
        elif winner == 0:
            reward = 0.5
        else:
            reward = -1
        return reward

    def learn(self, prev_state, prev_move, state, available_moves, move, reward): #Online Learning
        if prev_move != -1:

            target = self.model.calc_target(prev_state, prev_move, state, available_moves, reward)
            self.model.train_model(prev_state, prev_move, target, 1)
    
    def load_to_memory(self, prev_state, prev_move, state, available_moves, reward):
        self.memory.append([prev_state, prev_move, state, available_moves, reward])

# Connect 4 MCTS

In [9]:
class MCTS:

    def __init__(self, state, tag, depth=0):
        self.state = state
        self.m1 = None
        self.m2 = None
        self.m3 = None
        self.m4 = None
        self.m5 = None
        self.m6 = None
        self.m7 = None
        self.prev = None

        self.tag = tag
        self.num_visit = 0
        self.num_win = 0
        self.depth = depth

    def expand(self, move, state, tag):
        sub_tree = MCTS(state, tag, self.depth + 1)
        setattr(self, 'm' + str(move), sub_tree)
        sub_tree.prev = self
        return sub_tree

# MCTS Agent

In [10]:
class MCTSAgent:

    def __init__(self, tag):
        self.tag = tag
        self.c = np.sqrt(2)/10
        self.play_tree = self.load_tree(tag)
        self.expand_flag = True
        global t
        global con_tree

    def choose_move(self, state, winner, learn, episode):
        self.expand_opp_move(state, learn)

        # expand tree to opponent move
        if winner is not None:
            self.back_prop_tree(winner)
            if learn is True:
                self.expand_flag = True
            return

        # check if all leaf init
        available_moves = ava_moves(state)
        idx = []

        for move in available_moves:
            if getattr(self.play_tree, 'm' + str(move+1)) is None:
                idx.append(move)

        if len(idx) > 0:
            leaf_init = False
        else:
            leaf_init = True

        # if true - pick Bandit
        if leaf_init is True:
            move = self.pick_bandit(available_moves)
            self.play_tree = getattr(self.play_tree, 'm' + str(move + 1))
            return move

        # if false - there is at least 1 None
        else:
            # if learn false pick random
            if learn is False:
                return random.choice(available_moves)
            # if learn true pick random from none and expand
            else:
                move = random.choice(idx)
                new_state = self.make_state_from_move(state, move)

                if self.tag == 1:
                    tag = 2
                else:
                    tag = 1

                if self.expand_flag is True:
                    self.play_tree = self.play_tree.expand(move + 1, new_state, tag)

            self.expand_flag = False
            return move

    def expand_opp_move(self, state, learn):
        if self.expand_flag is False:
            return

        prev_state = self.play_tree.state
        diff = prev_state - state
        _, idx = np.nonzero(diff)

        if len(idx) == 0:
            return

        idx = idx[0]

        if self.tag == 1:
            opp_tag = 2
        else:
            opp_tag = 1

        if getattr(self.play_tree, 'm' + str(idx+1)) is None:
            if learn is False:
                pass
            else:
                self.play_tree = self.play_tree.expand(idx+1, state, opp_tag)
        else:
            self.play_tree = getattr(self.play_tree, 'm' + str(idx + 1))

    def back_prop_tree(self, winner):
        while self.play_tree.prev is not None:
            if winner == self.tag:
                self.play_tree.num_win += 1

            self.play_tree.num_visit += 1

            self.play_tree = self.play_tree.prev

        if winner == self.tag:
            self.play_tree.num_win += 1
        self.play_tree.num_visit += 1

        return

    def pick_bandit(self, available_moves):
        mct = 0
        idx = []

        for move in available_moves:
            num_vis = getattr(getattr(self.play_tree, 'm' + str(move + 1)), 'num_visit')
            num_w = getattr(getattr(self.play_tree, 'm' + str(move + 1)), 'num_win')

            total_num = self.play_tree.num_visit

            value = num_w / num_vis + self.c * np.sqrt(np.log(total_num / num_vis))

            if value > mct:
                mct = value
                idx = [move]
            elif value == mct:
                idx.append(move)

        return random.choice(idx)

    def make_state_from_move(self, state, move):
        if self.tag == 1:
            tag = 1
        else:
            tag = -1

        idy = np.where(state[:, move] == 0)[0][-1]
        new_state = np.array(state)
        new_state[idy, move] = tag
        return new_state

    def save_tree(self):
        tag = tag = firstOrSecond(self.tag)
        s = 'MCTS Agents/MCTS_Agent_Going_' + str(tag) + '.pkl'

        try:
            os.remove(s)
        except:
            pass

        with open(s, 'wb') as output:
            pickle.dump(self.play_tree, output)
            
    @staticmethod
    def load_tree(tag):
        tag = firstOrSecond(tag)
        s = 'MCTS Agents/MCTS_Agent_Going_' + str(tag) + '.pkl'
        print(s)
        tree_file = Path(s)
        if tree_file.is_file():
            print('load tree tag:', tag)
            with open(s, 'rb') as input_:
                tr = pickle.load(input_)
                return tr
        else:
            print('new tree tag:', tag)
            return MCTS(np.zeros((6, 7)), 1)

# Random Agent

In [11]:
class RandomAgent:

    def __init__(self, tag):
        self.tag = tag

    @staticmethod
    def choose_move(state, winner, learn, episode):
        if winner is not None:
            return
        
        moves = ava_moves(state)
        return random.choice(moves)

# Train

In [39]:
episodes = 1800
#episodes = 10000
#episodes = 100000
#episodes = 1000000

In [40]:
#Player 1 Agents

#Random Agent
agent1 = RandomAgent(tag=1)

#MCTS Agent
#agent1 = MCTSAgent(tag=1) 

#Q-Learning Agent, Fixed Epsilon, Network A
#agent1 = ConnectFourAgent(tag=1, starting_epsilon=0.2, constant_epsilon=True, max_memory=300, network='A') 

#Q-Learning Agent, Fixed Epsilon, Network B
#agent1 = ConnectFourAgent(tag=1, starting_epsilon=0.2, constant_epsilon=True, max_memory=300, network='B') 

#Q-Learning Agent, Fixed Epsilon 0.1, Network B
#agent1 = ConnectFourAgent(tag=1, starting_epsilon=0.1, constant_epsilon=True, max_memory=300, network='B') 

#Q-Learning Agent, non-fixed Epsilon, Network A
#agent1 = ConnectFourAgent(tag=1, starting_epsilon=1, constant_epsilon=False, max_memory=300, network='A')

#Q-Learning Agent, non-fixed Epsilon, Network B
#agent1 = ConnectFourAgent(tag=1, starting_epsilon=1, constant_epsilon=False, max_memory=300, network='B')

In [41]:
#Player 2 Agents

#Random Agent
agent2 = RandomAgent(tag=2)

#MCTS Agent
#agent2 = MCTSAgent(tag=2) 

#Q-Learning Agent, Fixed Epsilon, Network A
#agent2 = ConnectFourAgent(tag=2, starting_epsilon=0.2, constant_epsilon=True, max_memory=300, network='A') 

#Q-Learning Agent, Fixed Epsilon, Network B
#agent2 = ConnectFourAgent(tag=2, starting_epsilon=0.2, constant_epsilon=True, max_memory=300, network='B') 

#Q-Learning Agent, Fixed Epsilon 0.1, Network B
#agent2 = ConnectFourAgent(tag=2, starting_epsilon=0.1, constant_epsilon=True, max_memory=300, network='B') 

#Q-Learning Agent, non-fixed Epsilon, Network A
#agent2 = ConnectFourAgent(tag=2, starting_epsilon=1, constant_epsilon=False, max_memory=300, network='A')

In [42]:
print( "Current Time = " + str( datetime.datetime.now() ) )

game = ConnectFour(agent1, agent2, episodes, learn=True, printGame=True)
statistics = game.play_multiple_games()
#print('Q-Learning Agent vs Random Agent', statistics)
#print('Q-Learning Agent vs MCTS Agent', statistics)
print('Q-Learning Agent vs Q-Learning Agent', statistics)
#print('Random Agent vs MCTS Agent', statistics)
#print('MCTS Agent vs Random Agent', statistics)

print( "Current Time = " + str( datetime.datetime.now() ) )

Current Time = 2021-12-11 20:16:42.956015
Game 1
winner is: 2
[[-1.  1.  0.  0.  1.  1. -1.]
 [-1. -1. -1.  0.  1. -1. -1.]
 [ 1.  1. -1.  0.  1. -1.  1.]
 [ 1.  1. -1.  1. -1. -1. -1.]
 [-1. -1. -1.  1. -1.  1.  1.]
 [ 1. -1.  1.  1.  1. -1.  1.]]
Game 2
winner is: 1
[[ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0. -1.  0.  0.]
 [ 0.  0.  0.  0.  1.  0.  0.]
 [-1.  1. -1.  0. -1.  0. -1.]
 [ 1.  1.  1.  1.  1.  0. -1.]]
Game 3
winner is: 1
[[ 0.  0.  1.  0.  0.  0.  1.]
 [ 0.  0. -1.  0.  0.  0.  1.]
 [ 0.  0.  1.  0.  0.  0.  1.]
 [ 0. -1. -1.  0.  0.  0.  1.]
 [-1. -1.  1. -1.  1. -1. -1.]
 [ 1.  1.  1. -1.  1. -1. -1.]]
Game 4
winner is: 2
[[ 0.  0.  0.  0.  0.  0.  0.]
 [-1.  0.  0.  0.  0.  0.  0.]
 [-1.  0.  0.  0.  0.  0.  0.]
 [-1.  1.  1.  0.  0.  0.  0.]
 [-1. -1. -1.  1.  0.  1. -1.]
 [ 1. -1.  1.  1.  1. -1.  1.]]
Game 5
winner is: 2
[[ 0.  1.  0.  0.  0.  0.  0.]
 [ 0.  1.  0.  0.  0. -1.  0.]
 [ 0. -1.  0.  0.  0. -1.  0.]
 [ 0.  1.  0.  0.

winner is: 2
[[ 0.  0.  0.  0.  0. -1.  0.]
 [ 0.  0.  0.  0.  0.  1.  0.]
 [ 0.  0.  0.  0.  0.  1. -1.]
 [-1.  0.  1.  0. -1. -1.  1.]
 [-1. -1.  1.  0. -1.  1. -1.]
 [ 1.  1.  1. -1.  1.  1. -1.]]
Game 52
winner is: 2
[[-1.  0.  0.  0.  0.  0.  0.]
 [-1.  0.  0.  0.  0.  0.  0.]
 [ 1.  0.  1. -1.  1.  1.  0.]
 [ 1. -1. -1. -1.  1. -1.  1.]
 [-1.  1.  1. -1.  1.  1. -1.]
 [ 1. -1.  1. -1. -1.  1. -1.]]
Game 53
winner is: 2
[[ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  1.  0.  0.  0.  0.  0.]
 [ 0. -1.  0.  0. -1.  0.  0.]
 [ 0.  1.  1.  0. -1.  0. -1.]
 [-1.  1. -1.  1. -1.  0.  1.]
 [ 1. -1. -1.  1. -1.  1.  1.]]
Game 54
winner is: 1
[[ 0.  0.  0.  0.  0.  0.  1.]
 [ 0.  0.  0.  0.  0.  0. -1.]
 [ 0.  0.  0.  0. -1.  0.  1.]
 [-1.  0.  0.  0.  1.  1.  1.]
 [ 1.  0. -1.  0.  1. -1. -1.]
 [-1.  0. -1.  1. -1.  1.  1.]]
Game 55
winner is: 1
[[ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0. -1.]


winner is: 2
[[ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [-1.  1.  0.  0.  1.  0.  0.]
 [-1.  1.  1.  0. -1. -1.  0.]
 [-1. -1. -1.  0.  1.  1.  1.]
 [-1.  1.  1.  0. -1. -1.  1.]]
Game 108
winner is: 2
[[ 0.  0.  0.  0.  0.  1.  0.]
 [ 0.  0. -1.  0.  0.  1.  1.]
 [ 0.  0.  1. -1.  0. -1. -1.]
 [ 1. -1. -1. -1.  0.  1. -1.]
 [-1.  1.  1. -1.  1.  1.  1.]
 [-1. -1.  1. -1.  1.  1. -1.]]
Game 109
winner is: 2
[[ 0.  0.  0.  0.  0.  0.  0.]
 [ 1.  0.  0.  0.  0.  0.  0.]
 [ 1.  0.  0.  1.  0.  0. -1.]
 [-1.  0.  0.  1.  0.  1. -1.]
 [ 1.  0.  1. -1.  0.  1. -1.]
 [-1.  1. -1. -1.  1. -1. -1.]]
Game 110
winner is: 1
[[ 0.  0. -1.  0.  0.  0.  0.]
 [ 0.  0. -1.  0.  0.  0.  0.]
 [ 0.  0.  1.  0.  0.  1.  0.]
 [ 0.  0. -1.  0.  0.  1.  0.]
 [ 0. -1.  1. -1. -1.  1.  1.]
 [ 1.  1.  1. -1. -1.  1. -1.]]
Game 111
winner is: 1
[[ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  1.  0.  1.  0.  0.]
 [ 0.  0. -1.  0. -1. -1. -1.]
 [ 0.  1. -1.  0. -1.  1.  

winner is: 1
[[ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 1.  0.  0.  0.  0.  0.  0.]
 [ 1.  1.  0.  1.  0. -1.  0.]
 [ 1.  1.  0. -1.  0. -1. -1.]
 [ 1. -1. -1.  1. -1. -1.  1.]]
Game 162
winner is: 2
[[ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0. -1.  0.  0.  0.]
 [-1.  1.  0. -1. -1.  0.  1.]
 [ 1.  1.  0. -1.  1.  0. -1.]
 [ 1.  1. -1. -1. -1.  1.  1.]]
Game 163
winner is: 2
[[ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0. -1.  0.  0.  0.  0.]
 [-1.  0. -1.  0.  1.  0.  0.]
 [ 1.  0. -1.  0.  1.  0.  0.]
 [-1.  1. -1.  0.  1.  0.  1.]]
Game 164
winner is: 2
[[ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0. -1.  1.]
 [ 0.  0.  0.  0.  1. -1. -1.]
 [ 1.  0.  0.  0. -1. -1. -1.]
 [ 1.  0.  0.  1.  1. -1.  1.]]
Game 165
winner is: 1
[[ 0.  1.  0.  0.  0. -1.  0.]
 [ 0. -1.  0.  0.  0.  1.  0.]
 [ 1.  1.  1.  0.  0.  1.  0.]
 [-1. -1.  1.  0. -1. -1.  0.]
 [-1.  1.  1.  0. -1. -1.  

winner is: 1
[[ 1.  0.  0.  0.  0.  0.  0.]
 [-1.  0.  0.  0.  0.  0.  1.]
 [-1.  1.  0.  0.  0. -1.  1.]
 [ 1.  1. -1. -1.  0. -1.  1.]
 [-1. -1.  1. -1. -1. -1.  1.]
 [ 1.  1. -1.  1.  1.  1. -1.]]
Game 215
winner is: 2
[[ 0.  0.  0.  0.  0.  0.  0.]
 [ 0. -1.  0.  0.  0.  0.  0.]
 [ 0. -1.  0.  0.  1.  0.  0.]
 [ 0.  1. -1. -1.  1. -1.  0.]
 [ 1. -1.  1. -1.  1.  1.  1.]
 [-1.  1. -1.  1. -1. -1.  1.]]
Game 216
winner is: 1
[[ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  1.  0.]
 [ 0.  0.  0.  0.  0. -1.  0.]
 [ 0.  0.  0.  0.  0.  1.  0.]
 [ 0.  0.  0.  0. -1. -1. -1.]
 [ 1.  1.  1.  1. -1.  1. -1.]]
Game 217
winner is: 2
[[ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0. -1.  0.  0.  0.  1.  0.]
 [ 1.  1. -1. -1. -1. -1.  1.]
 [ 1.  1. -1. -1.  1.  1. -1.]]
Game 218
winner is: 2
[[ 0.  0.  0.  0.  0.  1.  0.]
 [ 1.  0.  0.  0.  1.  1.  0.]
 [ 1. -1.  0.  0. -1. -1.  0.]
 [ 1. -1. -1. -1.  1. -1.  0.]
 [-1. -1.  1. -1. -1.  1.  

...

winner is: 2
[[ 0.  0.  0.  0.  0.  0.  0.]
 [-1.  1.  0.  0.  0.  0.  0.]
 [-1. -1.  0.  0.  0.  0.  0.]
 [ 1.  1. -1.  0.  1.  0.  0.]
 [ 1. -1.  1. -1.  1. -1. -1.]
 [ 1. -1. -1.  1.  1.  1. -1.]]
Game 1559
winner is: 1
[[ 0.  0.  0.  0.  0.  0.  0.]
 [ 1.  0.  0.  0.  0.  0.  0.]
 [-1.  0.  0.  0.  0.  1. -1.]
 [ 1. -1.  0. -1.  0. -1. -1.]
 [ 1. -1.  1.  1.  1.  1. -1.]
 [-1. -1.  1.  1.  1. -1.  1.]]
Game 1560
winner is: 2
[[ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [-1.  0.  0.  0.  0.  0.  0.]
 [-1.  0.  1.  0.  1.  0.  0.]
 [-1.  0.  1.  0.  1.  0.  1.]
 [ 1.  0. -1. -1. -1. -1.  1.]]
Game 1561
winner is: 2
[[ 0.  0.  0.  0.  0.  0.  0.]
 [-1.  0. -1.  0. -1.  0.  0.]
 [ 1. -1. -1.  0.  1.  0.  0.]
 [ 1.  1. -1.  1.  1.  0.  0.]
 [-1. -1. -1.  1. -1.  1.  1.]
 [ 1.  1.  1. -1.  1. -1. -1.]]
Game 1562
winner is: 1
[[ 0.  0.  0.  0.  0. -1.  0.]
 [ 0.  0.  1.  0.  0. -1.  0.]
 [ 0.  0. -1.  1.  0.  1.  0.]
 [-1.  0.  1. -1.  0.  1. -1.]
 [ 1.  1. -1.  1.  1.  

winner is: 1
[[ 0.  0.  0.  0.  0.  0.  0.]
 [-1.  0.  0.  0.  0. -1.  0.]
 [ 1.  0.  0.  0.  0. -1.  1.]
 [-1. -1.  0.  0.  1.  1.  1.]
 [-1.  1. -1.  0.  1. -1. -1.]
 [-1.  1.  1.  1. -1.  1.  1.]]
Game 1613
winner is: 1
[[ 0.  0.  0. -1.  0.  0.  0.]
 [ 0.  0.  0.  1.  0.  0.  0.]
 [-1.  0.  0.  1.  0.  0.  0.]
 [-1.  0. -1. -1.  1.  0.  0.]
 [ 1.  0. -1.  1.  1.  1. -1.]
 [-1.  1.  1. -1.  1. -1.  1.]]
Game 1614
winner is: 1
[[ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  1.  0.  0.  0.  0.  0.]
 [-1. -1.  0.  0.  0.  1.  1.]
 [-1.  1. -1.  0.  1. -1.  1.]
 [-1. -1.  1.  1. -1.  1. -1.]
 [ 1. -1.  1.  1.  1. -1. -1.]]
Game 1615
winner is: 2
[[ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  1.  0.  0.  0.]
 [ 0.  0.  1. -1. -1.  0.  0.]
 [-1.  0. -1.  1. -1.  1. -1.]
 [ 1.  1.  1. -1. -1. -1.  1.]
 [ 1. -1.  1. -1. -1.  1.  1.]]
Game 1616
winner is: 1
[[ 0.  1. -1.  0.  0.  0.  0.]
 [ 0.  1. -1.  0.  0.  0.  0.]
 [-1.  1. -1.  0.  0.  0.  0.]
 [ 1. -1.  1.  0.  0.  0.  0.]
 [ 1. -1.  1.  1. -1.  

winner is: 1
[[ 0.  0.  1.  0.  0.  0.  0.]
 [ 0.  0. -1.  0.  0.  0. -1.]
 [ 0. -1.  1. -1.  0.  0. -1.]
 [ 1.  1.  1.  1.  0.  1. -1.]
 [-1. -1. -1.  1. -1.  1.  1.]
 [ 1.  1.  1. -1. -1. -1.  1.]]
Game 1667
winner is: 2
[[ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  1.  0.  0.  0.  0.]
 [ 0.  0.  1.  0.  0.  0.  0.]
 [ 1.  0. -1. -1. -1. -1.  0.]
 [ 1.  0. -1.  1.  1. -1. -1.]
 [ 1.  1. -1. -1.  1. -1.  1.]]
Game 1668
winner is: 1
[[ 0.  0.  0. -1.  0.  0.  0.]
 [ 0.  0. -1.  1.  0.  0.  0.]
 [ 0.  0.  1. -1.  0.  0.  0.]
 [ 0.  1. -1.  1.  1. -1.  0.]
 [ 1. -1.  1. -1. -1.  1.  0.]
 [ 1. -1. -1.  1.  1. -1.  1.]]
Game 1669
winner is: 2
[[ 0.  0.  0.  0. -1.  0.  0.]
 [ 0.  0.  0.  0. -1.  0.  0.]
 [ 0.  0. -1.  0.  1.  0.  0.]
 [ 0.  0.  1. -1.  1. -1.  0.]
 [-1.  1.  1.  1. -1. -1.  1.]
 [ 1.  1. -1.  1. -1. -1.  1.]]
Game 1670
winner is: 1
[[ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0. -1.  0.  0.  0.  0.  1.]
 [ 0. -1.  0.  1.  0.  

winner is: 1
[[ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  1.  0.  0.  0.  0.  0.]
 [ 0. -1.  1.  1.  0.  0.  0.]
 [ 0. -1.  1.  1.  0.  0.  0.]
 [ 0. -1.  1. -1.  1. -1. -1.]]
Game 1716
winner is: 2
[[ 0.  0.  0.  0.  1.  1.  0.]
 [ 1.  0.  0.  0.  1.  1. -1.]
 [ 1.  0.  0. -1. -1.  1. -1.]
 [-1. -1.  1. -1. -1. -1.  1.]
 [-1. -1.  1.  1.  1. -1. -1.]
 [ 1. -1.  1. -1.  1.  1. -1.]]
Game 1717
winner is: 2
[[ 0.  0.  1.  0.  0.  0. -1.]
 [-1.  0.  1.  0.  0.  1.  1.]
 [-1.  0. -1.  0.  0. -1. -1.]
 [-1.  1. -1. -1.  0.  1. -1.]
 [-1. -1.  1.  1.  1. -1.  1.]
 [ 1.  1.  1. -1.  1.  1. -1.]]
Game 1718
winner is: 1
[[ 1.  1.  0. -1.  0.  0.  0.]
 [ 1. -1.  0.  1.  0. -1.  0.]
 [-1.  1.  0. -1.  0.  1.  1.]
 [-1. -1.  0.  1. -1.  1. -1.]
 [ 1. -1.  0. -1.  1. -1.  1.]
 [-1. -1.  1.  1. -1.  1.  1.]]
Game 1719
winner is: 2
[[ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0. -1.  0.  0.  0.  0.  0.]
 [ 1. -1.  0.  0.  0.  0.  0.]
 [ 1. -1.  0.  0.  1.  

winner is: 2
[[ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  1.  0.  1.  0. -1.  0.]
 [ 0.  1. -1. -1.  0.  1.  0.]
 [ 0. -1.  1. -1. -1. -1. -1.]
 [ 1.  1. -1. -1.  1.  1.  1.]]
Game 1776
winner is: 2
[[ 0.  0.  0.  1.  0.  0.  0.]
 [ 0.  0.  0.  1.  0.  0.  1.]
 [ 0.  1.  0. -1.  0.  0.  1.]
 [ 0. -1.  0. -1. -1. -1. -1.]
 [ 0. -1.  1.  1. -1. -1.  1.]
 [ 1.  1. -1. -1. -1.  1.  1.]]
Game 1777
winner is: 2
[[ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0. -1.  0.  1.  0.  0.  0.]
 [ 0. -1.  0.  1.  0.  0. -1.]
 [-1. -1.  0.  1.  1.  0.  1.]
 [ 1. -1.  1. -1.  1.  0. -1.]]
Game 1778
winner is: 1
[[ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0. -1.  0.  1. -1.  0.  0.]
 [ 0. -1. -1.  1.  1.  1.  1.]]
Game 1779
winner is: 2
[[ 0.  0.  0.  0.  0.  0.  0.]
 [ 0. -1.  0.  0.  0.  0.  0.]
 [ 0.  1.  1.  0.  0. -1. -1.]
 [ 1.  1.  1.  0.  0. -1.  1.]
 [-1. -1. -1.  1. -1. -

# Load and Test

In [352]:
#episodes = 100
episodes = 10000

In [353]:
#Random Agent
#agent1 = RandomAgent(tag=1)

#MCTS Agent
agent1 = MCTSAgent(tag=1) 

#Q-Learning Agent, Fixed Epsilon
#agent1 = ConnectFourAgent(tag=1, starting_epsilon=0, constant_epsilon=True, max_memory=0, network='A')

MCTS Agents/MCTS_Agent_Going_First.pkl
load tree tag: First


In [354]:
#Player 2 Agents

#Random Agent
agent2 = RandomAgent(tag=2)

#MCTS Agent
#agent2 = MCTSAgent(tag=2) 

#Q-Learning Agent, Fixed Epsilon
#agent2 = ConnectFourAgent(tag=2, starting_epsilon=0, constant_epsilon=True, max_memory=0, network='A')

In [355]:
#printGame2=True
printGame2=False

In [356]:
totalWinsPlayer1 = 0
totalWinsPlayer2 = 0
totalDraws = 0
totalNumberOfMoves = 0

for seed in seeds:
    random.seed(seed)

    game = ConnectFour(agent1, agent2, episodes, learn=False, printGame=printGame2)
    statistics = game.play_multiple_games()
    print('Seed ' + str(seed), statistics)
    totalWinsPlayer1 += statistics[1]
    totalWinsPlayer2 += statistics[2]
    totalDraws += statistics[0]
    totalNumberOfMoves += statistics['Number of Moves']
    
#print('Mean Number of Wins = ' + str(totalWinsPlayer1/seedsSize))
#print('Mean Number of Losses = ' + str(totalWinsPlayer2/seedsSize))
#print('Mean Number of Draws = ' + str(totalDraws/seedsSize))
#print('Mean Number of Moves = ' + str(totalNumberOfMoves/seedsSize))

Seed 68662 {1: 8848, 2: 1139, 0: 13, 'Number of Moves': 12.0636}


# Play Against it

In [15]:
class HumanPlayer:

    def __init__(self, tag):
        self.tag = tag

    @staticmethod
    def choose_move(state, winner, learn, episode):
        if winner is not None:
            return
        
        print(state)
        move = int(input('Choose move number: ')) - 1
        return move

In [16]:
episodes = 1

In [19]:
#Agents

#Q-Learning Agent, Fixed Epsilon
#agent1 = ConnectFourAgent(tag=1, starting_epsilon=0, constant_epsilon=True, max_memory=0, network='A')

#MCTS Agent
agent1 = MCTSAgent(tag=1) 

#Human Player
agent2 = HumanPlayer(tag=2) 

MCTS Agents/MCTS_Agent_Going_First.pkl
load tree tag: First


In [20]:
game = ConnectFour(agent1, agent2, episodes, learn=False, printGame=True)
statistics = game.play_multiple_games()
print('Q-Learning Agent vs Human Player', statistics)

Game 1
[[0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0.]]
Choose move number: 4
[[ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  1.  0.  0.  0.  0.]
 [ 0.  0.  1. -1.  0.  0.  0.]]
Choose move number: 3
[[ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0. -1.  0.  0.  0.  0.]
 [ 0.  0.  1.  0.  0.  0.  0.]
 [ 0.  0.  1. -1.  0.  0.  1.]]
Choose move number: 4
[[ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0. -1.  1.  0.  0.  0.]
 [ 0.  0.  1. -1.  0.  0.  0.]
 [ 0.  0.  1. -1.  0.  0.  1.]]
Choose move number: 5
[[ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0. -1.  1.  0.  0.  0.]
 [ 0.  0.  1. -1.  0.  0.  1.]
 [ 0.  0.  1. -1. -1.  0.  1.]]
Choose move 